In [40]:
energy_features = [
    "energy_mean",
    "energy_std",
    "energy_max",
    "energy_slope",
    "time_to_stillness"
]

all_features = [
    "energy_mean",
    "energy_std",
    "energy_min",
    "energy_max",
    "energy_slope",
    "stillness_ratio",
    "time_to_stillness",
    "doppler_centroid_mean",
    "doppler_centroid_std",
    "doppler_spread_mean",
    "doppler_spread_std",
    "peak_range_mode",
    "peak_range_std",
    "peak_range_changes",
    "range_spread_mean",
    "range_spread_std"
]

In [41]:
import numpy as np
import pandas as pd
import joblib
import os

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score

FEATURES_PATH = "/kaggle/input/datasets/jorisha/radar-data2/ALL_phase3_features.npz"
META_PATH = "/kaggle/input/datasets/jorisha/radar-data2/ALL_phase3_metadata.csv"

data = np.load(FEATURES_PATH, allow_pickle=True)
X_all = data["X"]

meta = pd.read_csv(META_PATH)

feature_cols = all_features

df_X = pd.DataFrame(X_all, columns=feature_cols)

mask = (
    (meta["y_t8"] != -1) &
    (meta["truncated"] == False) &
    (meta["skipped"] == False)
)

df = pd.concat([df_X, meta], axis=1)
df_clean = df[mask].reset_index(drop=True)

y = df_clean["y_t8"].astype(int).values
groups = df_clean["subject_id"].astype(str).values

print("Clean samples:", len(df_clean))
print(pd.Series(y).map({0:"Low", 1:"High"}).value_counts())

Clean samples: 117
Low     87
High    30
Name: count, dtype: int64


In [42]:
from sklearn.metrics import f1_score, recall_score, precision_score

def percentile_baseline(df_clean, feature_name, direction="high"):
    X = df_clean[[feature_name]].values
    y = df_clean["y_t8"].astype(int).values
    groups = df_clean["subject_id"].astype(str).values

    gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
    train_idx, test_idx = next(gss.split(X, y, groups=groups))

    train_values = df_clean.iloc[train_idx][feature_name].values
    test_values = df_clean.iloc[test_idx][feature_name].values
    y_test = y[test_idx]
    y_train = y[train_idx]

    high_ratio = np.mean(y_train == 1)
    percentile = 100 * (1 - high_ratio)

    if direction == "high":
        threshold = np.percentile(train_values, percentile)
        y_pred = (test_values >= threshold).astype(int)
    else:
        threshold = np.percentile(train_values, 100 - percentile)
        y_pred = (test_values <= threshold).astype(int)

    print(f"\n===== Percentile baseline: {feature_name} / {direction} =====")
    print("Threshold:", threshold)
    print(classification_report(
        y_test, y_pred,
        target_names=["Low", "High"],
        zero_division=0
    ))
    print(confusion_matrix(y_test, y_pred))

    return threshold

In [43]:
import numpy as np
import pandas as pd
import joblib, os, json

from sklearn.model_selection import StratifiedKFold, GroupKFold, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score, f1_score, recall_score, precision_score

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

In [44]:
# ===============================
# Prepare data
# ===============================

FEATURE_SET = all_features   # جربي بعدها energy_features

X = df_clean[FEATURE_SET].values
y = df_clean["y_t8"].astype(int).values   # 0 Low, 1 High
groups = df_clean["subject_id"].astype(str).values

print("Data:", X.shape)
print(pd.Series(y).map({0:"Low", 1:"High"}).value_counts())

Data: (117, 16)
Low     87
High    30
Name: count, dtype: int64


In [45]:
# ===============================
# Candidate models
# ===============================

models = {
    "LogisticRegression": LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    ),

    "RandomForest": RandomForestClassifier(
        n_estimators=500,
        max_depth=4,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42
    ),

    "ExtraTrees": ExtraTreesClassifier(
        n_estimators=500,
        max_depth=4,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42
    ),

    "SVM_RBF": SVC(
        kernel="rbf",
        class_weight="balanced",
        probability=True,
        C=1,
        gamma="scale",
        random_state=42
    ),

    "KNN": KNeighborsClassifier(
        n_neighbors=5,
        weights="distance"
    ),

    "GradientBoosting": GradientBoostingClassifier(
        random_state=42
    ),

    "AdaBoost": AdaBoostClassifier(
        n_estimators=200,
        random_state=42
    ),

    "GaussianNB": GaussianNB()
}

In [49]:
# ============================================
# Leakage-Free Radar Severity Evaluation
# ============================================

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

import numpy as np
import pandas as pd

# ============================================
# Threshold tuning ONLY on validation
# ============================================

def tune_threshold(probs, y_true):

    rows = []

    for t in np.arange(0.10, 0.91, 0.05):

        y_pred = (probs >= t).astype(int)

        rows.append({
            "threshold": round(t, 2),
            "balanced_acc": balanced_accuracy_score(y_true, y_pred),
            "high_precision": precision_score(y_true, y_pred, zero_division=0),
            "high_recall": recall_score(y_true, y_pred, zero_division=0),
            "high_f1": f1_score(y_true, y_pred, zero_division=0),
        })

    df_thr = pd.DataFrame(rows)

    best = df_thr.sort_values(
        ["high_f1", "balanced_acc", "high_recall"],
        ascending=False
    ).iloc[0]

    return best["threshold"], df_thr


# ============================================
# Main evaluation
# ============================================

all_results = []
best_models = []

for seed in range(10):

    # ====================================
    # STEP 1:
    # train+val  vs test
    # ====================================

    gss_outer = GroupShuffleSplit(
        n_splits=1,
        test_size=0.25,
        random_state=seed
    )

    trainval_idx, test_idx = next(
        gss_outer.split(X, y, groups=groups)
    )

    X_trainval = X[trainval_idx]
    y_trainval = y[trainval_idx]

    X_test = X[test_idx]
    y_test = y[test_idx]

    groups_trainval = groups[trainval_idx]

    # skip bad test split
    if len(np.unique(y_test)) < 2:
        continue

    # ====================================
    # STEP 2:
    # split train vs validation
    # ====================================

    gss_inner = GroupShuffleSplit(
        n_splits=1,
        test_size=0.20,
        random_state=seed
    )

    train_idx, val_idx = next(
        gss_inner.split(
            X_trainval,
            y_trainval,
            groups=groups_trainval
        )
    )

    X_train = X_trainval[train_idx]
    y_train = y_trainval[train_idx]

    X_val = X_trainval[val_idx]
    y_val = y_trainval[val_idx]

    # skip bad val split
    if len(np.unique(y_val)) < 2:
        continue

    # ====================================
    # Try all models
    # ====================================

    for name, model in models.items():

        try:

            pipe = Pipeline([
                ("scaler", StandardScaler()),
                ("model", model)
            ])

            # ===============================
            # Train ONLY on train
            # ===============================

            pipe.fit(X_train, y_train)

            # ===============================
            # VALIDATION:
            # choose threshold here ONLY
            # ===============================

            if hasattr(pipe.named_steps["model"], "predict_proba"):

                val_probs = pipe.predict_proba(X_val)[:, 1]

            else:

                val_scores = pipe.decision_function(X_val)

                val_probs = (
                    val_scores - val_scores.min()
                ) / (
                    val_scores.max() - val_scores.min() + 1e-8
                )

            best_t, thr_table = tune_threshold(
                val_probs,
                y_val
            )

            # ===============================
            # TEST:
            # final evaluation ONLY
            # ===============================

            if hasattr(pipe.named_steps["model"], "predict_proba"):

                test_probs = pipe.predict_proba(X_test)[:, 1]

            else:

                test_scores = pipe.decision_function(X_test)

                test_probs = (
                    test_scores - test_scores.min()
                ) / (
                    test_scores.max() - test_scores.min() + 1e-8
                )

            y_pred = (test_probs >= best_t).astype(int)

            result = {
                "seed": seed,
                "model": name,
                "threshold": best_t,

                "val_low": int((y_val == 0).sum()),
                "val_high": int((y_val == 1).sum()),

                "test_low": int((y_test == 0).sum()),
                "test_high": int((y_test == 1).sum()),

                "balanced_acc": balanced_accuracy_score(y_test, y_pred),

                "high_precision": precision_score(
                    y_test,
                    y_pred,
                    zero_division=0
                ),

                "high_recall": recall_score(
                    y_test,
                    y_pred,
                    zero_division=0
                ),

                "high_f1": f1_score(
                    y_test,
                    y_pred,
                    zero_division=0
                ),
            }

            all_results.append(result)

            best_models.append({
                "result": result,
                "pipe": pipe,
                "threshold": best_t,
                "y_test": y_test,
                "y_pred": y_pred
            })

        except Exception as e:

            print("ERROR:", name, seed, e)

# ============================================
# Final Results
# ============================================

df_results = pd.DataFrame(all_results)

display(
    df_results.sort_values(
        ["high_f1", "balanced_acc"],
        ascending=False
    ).head(20)
)

# ============================================
# Best model
# ============================================

best_record = df_results.sort_values(
    ["high_f1", "balanced_acc"],
    ascending=False
).iloc[0]

print("\nBEST MODEL")
print(best_record)

,seed,model,threshold,val_low,val_high,test_low,test_high,balanced_acc,high_precision,high_recall,high_f1
33,4,RandomForest,0.30,9,4,18,8,0.909722,0.875000,0.875000,0.875000
34,4,ExtraTrees,0.50,9,4,18,8,0.875000,1.000000,0.750000,0.857143
1,0,RandomForest,0.15,8,14,17,8,0.911765,0.727273,1.000000,0.842105
22,2,AdaBoost,0.10,16,1,5,13,0.500000,0.722222,1.000000,0.838710
26,3,ExtraTrees,0.55,15,1,8,13,0.822115,0.909091,0.769231,0.833333
3,0,SVM_RBF,0.10,8,14,17,8,0.878676,0.777778,0.875000,0.823529
10,1,ExtraTrees,0.35,9,4,13,21,0.750916,0.809524,0.809524,0.809524
4,0,KNN,0.10,8,14,17,8,0.882353,0.666667,1.000000,0.800000
30,3,AdaBoost,0.45,15,1,8,13,0.649038,0.705882,0.923077,0.800000
25,3,RandomForest,0.50,15,1,8,13,0.783654,0.900000,0.692308,0.782609



BEST MODEL
seed                         4
model             RandomForest
threshold                  0.3
val_low                      9
val_high                     4
test_low                    18
test_high                    8
balanced_acc          0.909722
high_precision           0.875
high_recall              0.875
high_f1                  0.875
Name: 33, dtype: object


In [46]:
# ============================================
# Leakage-Free Radar Severity Evaluation
# ============================================

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score
)

import numpy as np
import pandas as pd

# ============================================
# Threshold tuning ONLY on validation
# ============================================

def tune_threshold(probs, y_true):

    rows = []

    for t in np.arange(0.10, 0.91, 0.05):

        y_pred = (probs >= t).astype(int)

        rows.append({
            "threshold": round(t, 2),
            "val_accuracy": accuracy_score(y_true, y_pred),
            "val_precision": precision_score(y_true, y_pred, average="binary", zero_division=0),
            "val_recall": recall_score(y_true, y_pred, average="binary", zero_division=0),
            "val_f1_score": f1_score(y_true, y_pred, average="binary", zero_division=0),
            "val_balanced_acc": balanced_accuracy_score(y_true, y_pred),
        })

    df_thr = pd.DataFrame(rows)

    best = df_thr.sort_values(
        ["val_f1_score", "val_balanced_acc", "val_recall"],
        ascending=False
    ).iloc[0]

    return best["threshold"], df_thr


# ============================================
# Main evaluation
# ============================================

all_results = []
best_models = []

for seed in range(10):

    # STEP 1: train+val vs test
    gss_outer = GroupShuffleSplit(
        n_splits=1,
        test_size=0.25,
        random_state=seed
    )

    trainval_idx, test_idx = next(
        gss_outer.split(X, y, groups=groups)
    )

    X_trainval = X[trainval_idx]
    y_trainval = y[trainval_idx]

    X_test = X[test_idx]
    y_test = y[test_idx]

    groups_trainval = groups[trainval_idx]

    if len(np.unique(y_test)) < 2:
        continue

    # STEP 2: train vs validation
    gss_inner = GroupShuffleSplit(
        n_splits=1,
        test_size=0.20,
        random_state=seed
    )

    train_idx, val_idx = next(
        gss_inner.split(
            X_trainval,
            y_trainval,
            groups=groups_trainval
        )
    )

    X_train = X_trainval[train_idx]
    y_train = y_trainval[train_idx]

    X_val = X_trainval[val_idx]
    y_val = y_trainval[val_idx]

    if len(np.unique(y_val)) < 2:
        continue

    for name, model in models.items():

        try:
            pipe = Pipeline([
                ("scaler", StandardScaler()),
                ("model", model)
            ])

            # Train ONLY on train
            pipe.fit(X_train, y_train)

            # Validation probabilities
            if hasattr(pipe.named_steps["model"], "predict_proba"):
                val_probs = pipe.predict_proba(X_val)[:, 1]
            else:
                val_scores = pipe.decision_function(X_val)
                val_probs = (
                    val_scores - val_scores.min()
                ) / (
                    val_scores.max() - val_scores.min() + 1e-8
                )

            # Choose threshold ONLY from validation
            best_t, thr_table = tune_threshold(val_probs, y_val)

            # Test probabilities
            if hasattr(pipe.named_steps["model"], "predict_proba"):
                test_probs = pipe.predict_proba(X_test)[:, 1]
            else:
                test_scores = pipe.decision_function(X_test)
                test_probs = (
                    test_scores - test_scores.min()
                ) / (
                    test_scores.max() - test_scores.min() + 1e-8
                )

            # Final test prediction
            y_pred = (test_probs >= best_t).astype(int)

            result = {

    "seed": seed,
    "model": name,
    "threshold": best_t,

    "val_low": int((y_val == 0).sum()),
    "val_high": int((y_val == 1).sum()),

    "test_low": int((y_test == 0).sum()),
    "test_high": int((y_test == 1).sum()),

    # ===== GENERAL MACRO METRICS =====

    "accuracy": accuracy_score(y_test, y_pred),

    "macro_precision": precision_score(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    ),

    "macro_recall": recall_score(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    ),

    "macro_f1": f1_score(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    ),

    "balanced_acc": balanced_accuracy_score(y_test, y_pred),
}

            all_results.append(result)

            best_models.append({
                "result": result,
                "pipe": pipe,
                "threshold": best_t,
                "y_test": y_test,
                "y_pred": y_pred,
                "thr_table": thr_table
            })

        except Exception as e:
            print("ERROR:", name, seed, e)


# ============================================
# Final Results Table
# ============================================

df_results = pd.DataFrame(all_results)

display(
    df_results.sort_values(
        ["macro_f1", "accuracy"],
        ascending=False
    ).head(20)
)


# ============================================
# Best model
# ============================================

best_record = df_results.sort_values(
    ["macro_f1", "accuracy", "balanced_acc"],
    ascending=False
).iloc[0]

print("\nBEST MODEL")
print(best_record)

,seed,model,threshold,val_low,val_high,test_low,test_high,accuracy,macro_precision,macro_recall,macro_f1,balanced_acc
33,4,RandomForest,0.30,9,4,18,8,0.923077,0.909722,0.909722,0.909722,0.909722
34,4,ExtraTrees,0.50,9,4,18,8,0.923077,0.950000,0.875000,0.902256,0.875000
1,0,RandomForest,0.15,8,14,17,8,0.880000,0.863636,0.911765,0.872666,0.911765
3,0,SVM_RBF,0.10,8,14,17,8,0.880000,0.857639,0.878676,0.866310,0.878676
36,4,KNN,0.45,9,4,18,8,0.884615,0.928571,0.812500,0.846154,0.812500
37,4,GradientBoosting,0.10,9,4,18,8,0.884615,0.928571,0.812500,0.846154,0.812500
4,0,KNN,0.10,8,14,17,8,0.840000,0.833333,0.882353,0.833333,0.882353
7,0,GaussianNB,0.10,8,14,17,8,0.840000,0.816176,0.816176,0.816176,0.816176
26,3,ExtraTrees,0.55,15,1,8,13,0.809524,0.804545,0.822115,0.805556,0.822115
35,4,SVM_RBF,0.15,9,4,18,8,0.807692,0.774510,0.791667,0.781513,0.791667



BEST MODEL
seed                          4
model              RandomForest
threshold                   0.3
val_low                       9
val_high                      4
test_low                     18
test_high                     8
accuracy               0.923077
macro_precision        0.909722
macro_recall           0.909722
macro_f1               0.909722
balanced_acc           0.909722
Name: 33, dtype: object


In [50]:
# ===============================
# Average model performance
# ===============================

summary = df_results.groupby("model").agg({
    "balanced_acc": ["mean", "std"],
    "high_precision": ["mean", "std"],
    "high_recall": ["mean", "std"],
    "high_f1": ["mean", "std"]
}).reset_index()

display(summary.sort_values(("high_f1", "mean"), ascending=False))

model balanced_acc           high_precision            \
                              mean       std           mean       std   
1          ExtraTrees     0.693611  0.147412       0.574524  0.321317   
6        RandomForest     0.712530  0.172842       0.594809  0.278863   
2          GaussianNB     0.686082  0.132125       0.551046  0.301533   
4                 KNN     0.658976  0.160218       0.577315  0.265763   
5  LogisticRegression     0.639700  0.107478       0.558082  0.291880   
0            AdaBoost     0.599943  0.145504       0.474652  0.237502   
7             SVM_RBF     0.653316  0.168328       0.470006  0.298540   
3    GradientBoosting     0.607765  0.171909       0.609970  0.227514   

  high_recall             high_f1            
         mean       std      mean       std  
1    0.822548  0.102200  0.611632  0.244995  
6    0.764296  0.226337  0.611017  0.241265  
2    0.669109  0.278931  0.563389  0.273855  
4    0.673331  0.305874  0.552443  0.233292  
5    0.667989  0.189063  0.535362  0.193141  
0    0.746998  0.261021  0.516513  0.230102  
7    0.587454  0.319435  0.481208  0.287943  
3    0.425977  0.276598  0.420005  0.163928

In [51]:
from sklearn.metrics import classification_report, confusion_matrix

best_item = sorted(
    best_models,
    key=lambda x: (
        x["result"]["high_f1"],
        x["result"]["balanced_acc"],
        x["result"]["high_recall"]
    ),
    reverse=True
)[0]

best_record = best_item["result"]
best_pipe = best_item["pipe"]
best_threshold = best_item["threshold"]
best_y_test = best_item["y_test"]
best_y_pred = best_item["y_pred"]

print("BEST:")
print(best_record)

print("\nClassification Report:")
print(classification_report(
    best_y_test,
    best_y_pred,
    target_names=["Low", "High"],
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(best_y_test, best_y_pred))

BEST:
{'seed': 4, 'model': 'RandomForest', 'threshold': np.float64(0.3), 'val_low': 9, 'val_high': 4, 'test_low': 18, 'test_high': 8, 'balanced_acc': np.float64(0.9097222222222222), 'high_precision': 0.875, 'high_recall': 0.875, 'high_f1': 0.875}

Classification Report:
              precision    recall  f1-score   support

         Low       0.94      0.94      0.94        18
        High       0.88      0.88      0.88         8

    accuracy                           0.92        26
   macro avg       0.91      0.91      0.91        26
weighted avg       0.92      0.92      0.92        26

Confusion Matrix:
[[17  1]
 [ 1  7]]
